# CSE 151B — Development notebook

Live in **`notebooks/dev.ipynb`**. Imports resolve **`REPO_ROOT`** automatically so `data/` and `results/` work whether the Jupyter cwd is the project root or `notebooks/`.

This notebook follows **`starter_code_cse151b_comp.ipynb`** end-to-end (vLLM inference → `Judger` scoring → JSONL output), but runs on a **held-out slice** of `public.jsonl` so you can iterate faster.

1. (**Colab**) Copy `public.jsonl` from Drive into `data/` (skip locally if the file already exists).
2. Build `data/dev.jsonl` (stratified MCQ / free-form, reproducible seed).
3. Same prompts, model load, generation loop, and scoring as the starter (`MAX_TOKENS` is 8192 here vs 4096 in starter).
4. Writes **`results/dev_results.jsonl`** (and a matching `.responses.jsonl` checkpoint).

Use the starter notebook for **full** public evaluation; use **`notebooks/submission.ipynb`** for private CSV export.

## 1. Environment (same as starter)

**Colab:** run the `%pip` cell below, restart runtime, continue. **Local:** use `uv` / your venv as in the starter — install the same packages (`vllm`, `transformers`, `bitsandbytes`, …).

In [1]:
# Colab: skip when working locally with an existing venv.
%pip install -q sympy numpy tqdm "antlr4-python3-runtime==4.11.1"
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu129 --upgrade
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.20.0/vllm-0.20.0+cu129-cp38-abi3-manylinux_2_31_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
%pip install -q transformers "bitsandbytes>=0.48.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 MB 2.4 MB/s eta 0:00:0000:0100:01


## 2. Imports & configuration

Matches the starter notebook; dev-specific keys:

- `PUBLIC_PATH` — full `public.jsonl` used only to **build** the dev file
- `DEV_PATH` — subset used as `DATA_PATH` for this run (`data/dev.jsonl`)
- `DEV_FRACTION` — fraction taken from **each** of MCQ and free-form (stratified)
- `DEV_SEED` — reproducible shuffle
- `OUTPUT_PATH` — default `results/dev_results.jsonl`

In [2]:
import json
import os
import random
from pathlib import Path


def _repo_root() -> Path:
    """Project root (parent of `data/`), whether cwd is repo root or `notebooks/`."""
    here = Path.cwd().resolve()
    if (here / "data").is_dir():
        return here
    if here.name == "notebooks" and (here.parent / "data").is_dir():
        return here.parent
    up = here.parent
    if (up / "data").is_dir():
        return up
    return here


REPO_ROOT = _repo_root()

# ── Dev split ─────────────────────────────────────────────────────────────────
PUBLIC_PATH   = str(REPO_ROOT / "data/public.jsonl")
DEV_PATH      = str(REPO_ROOT / "data/dev.jsonl")
DEV_FRACTION  = 0.10              # 10% per stratum (MCQ and free-form)
DEV_SEED      = 42
REBUILD_DEV   = True              # Set False to reuse existing DEV_PATH on disk

# ── Same as starter ───────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"
DATA_PATH   = DEV_PATH            # notebook loads the dev slice
OUTPUT_PATH = str(REPO_ROOT / "results/dev_results.jsonl")
MAX_TOKENS  = 8192              # 2× starter default (4096)

print(f"REPO_ROOT={REPO_ROOT}")

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import glob
import site

def _prepend_nvidia_libs_to_ld_path() -> None:
    roots = list(site.getsitepackages())
    user_site = getattr(site, "getusersitepackages", lambda: None)()
    if user_site:
        roots.append(user_site)
    libdirs: list[str] = []
    seen: set[str] = set()
    for root in roots:
        for d in glob.glob(os.path.join(root, "nvidia", "*", "lib")):
            if os.path.isdir(d) and d not in seen:
                seen.add(d)
                libdirs.append(d)
    if libdirs:
        sep = os.pathsep
        os.environ["LD_LIBRARY_PATH"] = sep.join(libdirs + [os.environ.get("LD_LIBRARY_PATH", "")]).strip(sep)


_prepend_nvidia_libs_to_ld_path()

import contextlib
import io
import re
import sys
from typing import Optional

@contextlib.contextmanager
def _jupyter_stdout_for_vllm():
    """Jupyter often replaces stdout with a stream that has no fileno(); vLLM workers need a real FD. Use only around vLLM load/generate so notebook print() still works elsewhere."""
    try:
        sys.stdout.fileno()
    except (io.UnsupportedOperation, AttributeError, OSError):
        saved_out, saved_err = sys.stdout, sys.stderr
        dup1, dup2 = os.dup(1), os.dup(2)
        try:
            sys.stdout = os.fdopen(dup1, "w", buffering=1)
            sys.stderr = os.fdopen(dup2, "w", buffering=1)
            yield
        finally:
            sys.stdout.close()
            sys.stderr.close()
            sys.stdout, sys.stderr = saved_out, saved_err
    else:
        yield

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

## 3. Colab: copy `public.jsonl` from Drive (same as starter)

On **Google Colab**, run this **before** building `dev.jsonl` if the VM has no repo files. Skip locally when `data/public.jsonl` already exists.

In [3]:
import shutil
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    print("Skip: not Google Colab — use repo `data/public.jsonl`.")
else:
    drive.mount("/content/drive")
    DRIVE_JSONL = Path("/content/drive/MyDrive/CSE151B/data/public.jsonl")
    DRIVE_PROJECT_ROOT = DRIVE_JSONL.parent.parent
    if not DRIVE_JSONL.is_file():
        raise FileNotFoundError(
            f"Missing on Drive: {DRIVE_JSONL}\n"
            "Upload public.jsonl or set DRIVE_JSONL."
        )
    (REPO_ROOT / "data").mkdir(parents=True, exist_ok=True)
    dest = REPO_ROOT / "data/public.jsonl"
    shutil.copy2(DRIVE_JSONL, dest)
    print(f"Copied to {dest.resolve()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied to /content/data/public.jsonl


## 4. Build `data/dev.jsonl` (stratified)

Samples `DEV_FRACTION` from MCQ rows and `DEV_FRACTION` from free-form rows separately so the dev mix stays close to public (~33% / ~67%). Sorts by `id` after selection.

In [4]:
def build_dev_jsonl(
    public_path: str,
    dev_path: str,
    fraction: float,
    seed: int,
) -> None:
    with open(public_path) as f:
        all_rows = [json.loads(line) for line in f]
    mcq  = [r for r in all_rows if r.get("options")]
    free = [r for r in all_rows if not r.get("options")]

    rng = random.Random(seed)

    def sample_frac(items: list) -> list:
        if not items:
            return []
        k = max(1, int(len(items) * fraction))
        k = min(k, len(items))
        idx = list(range(len(items)))
        rng.shuffle(idx)
        pick = idx[:k]
        return [items[i] for i in pick]

    dev_mcq  = sample_frac(mcq)
    dev_free = sample_frac(free)
    dev_rows = dev_mcq + dev_free
    dev_rows.sort(key=lambda r: r.get("id", 0))

    Path(dev_path).parent.mkdir(parents=True, exist_ok=True)
    with open(dev_path, "w") as out:
        for row in dev_rows:
            out.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(f"Wrote {len(dev_rows)} rows to {dev_path}")
    print(f"  MCQ in dev: {len(dev_mcq)} / {len(mcq)}   Free-form in dev: {len(dev_free)} / {len(free)}")
    print(f"  Public total: {len(all_rows)}")


if REBUILD_DEV or not Path(DEV_PATH).is_file():
    build_dev_jsonl(PUBLIC_PATH, DEV_PATH, DEV_FRACTION, DEV_SEED)
else:
    print(f"Using existing {DEV_PATH} (set REBUILD_DEV=True to resample)")

Wrote 112 rows to data/dev.jsonl
  MCQ in dev: 37 / 375   Free-form in dev: 75 / 751
  Public total: 1126


## 5. Load dataset

Uses `DATA_PATH` (`data/dev.jsonl`). Re-run the **build** cell if you change `DEV_FRACTION` / `DEV_SEED`.

In [5]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options") for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form) from {DATA_PATH}")

mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2)[:1200], "...\n" if len(json.dumps(mcq_sample)) > 1200 else "\n")
print("── Free-form sample ──")
print(json.dumps(free_sample, indent=2)[:1200], "...\n" if len(json.dumps(free_sample)) > 1200 else "\n")

Loaded 112 questions  (37 MCQ, 75 free-form) from data/dev.jsonl

── MCQ sample ──
{
  "question": "What is the center and radius of circle? $$\\left\\{\\begin{aligned} {{{}}} & {{} {{} {{} x^{2}+y^{2}+z^{2}=4}}} \\\\ {{{}}} & {{} {{{} x^{2}+y^{2}+z^{2}+x+y+z-7=0}}} \\\\ \\end{aligned} \\right.$$",
  "options": [
    "$$( 2, 1, 1 ), r=4$$",
    "$$( 2, 1, 2 ), r=5$$",
    "$$( 1, 1, 1 ), r=1$$",
    "$$( 1, 0, 1 ), r=2$$",
    "$$( 1, 1, 2 ), r=2$$",
    "$$( 0, 1, 1 ), r=1$$",
    "$$( 1, 1, 0 ), r=3$$",
    "$$( 0, 0, 1 ), r=4$$",
    "$$( 1, 2, 1 ), r=3$$",
    "$$( 1, 0, 0 ), r=5$$"
  ],
  "answer": "C",
  "id": 94
} 

── Free-form sample ──
{
  "question": "Reduce the fraction ${\\frac{25}{40}}$. [ANS]",
  "answer": [
    "5/8"
  ],
  "id": 3
} 



## 6. Prompt construction (starter + `docs/improvement-directions.md` §1.3 for MCQ)

In [6]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
What is the center and radius of circle? $$\left\{\begin{aligned} {{{}}} & {{} {{} {{} x^{2}+y^{2}+z^{2}=4}}} \\ {{{}}} & {{} {{{} x^{2}+y^{2}+z^{2}+x+y+z-7=0}}} \\ \end{aligned} \right.$$

Options:
A ...

── Free-form user prompt (first 200 chars) ──
Reduce the fraction ${\frac{25}{40}}$. [ANS] ...



## 7. Load model with vLLM (same as starter)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

with _jupyter_stdout_for_vllm():
    llm = LLM(
        model=MODEL_ID,
        quantization="bitsandbytes",
        load_format="bitsandbytes",
        enable_prefix_caching=True,
        gpu_memory_utilization=0.88,
        max_model_len=16384,
        trust_remote_code=True,
        max_num_seqs=64,
        max_num_batched_tokens=16384,
    )

sampling_params_free = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


INFO 05-02 20:47:08 [utils.py:233] non-default args: {'trust_remote_code': True, 'load_format': 'bitsandbytes', 'max_model_len': 16384, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.88, 'max_num_batched_tokens': 16384, 'max_num_seqs': 64, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'model': 'Qwen/Qwen3-4B-Thinking-2507'}
WARNING 05-02 20:47:08 [arg_utils.py:1467] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 05-02 20:47:09 [model.py:555] Resolved architecture: Qwen3ForCausalLM
INFO 05-02 20:47:09 [model.py:1680] Using max model len 16384
INFO 05-02 20:47:09 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 05-02 20:47:10 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 05-02 20:47:10 [nixl_utils.py:34] NIXL is not available
WARN

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 05-02 20:47:20 [gpu_model_runner.py:4879] Model loading took 2.7 GiB memory and 5.949236 seconds
INFO 05-02 20:47:26 [backends.py:1069] Using cache directory: /root/.cache/vllm/torch_compile_cache/95bbff7442/rank_0_0/backbone for vLLM's torch.compile
INFO 05-02 20:47:26 [backends.py:1128] Dynamo bytecode transform time: 4.79 s
INFO 05-02 20:47:28 [backends.py:290] Directly load the compiled graph(s) for compile range (1, 16384) from the cache, took 2.331 s
INFO 05-02 20:47:28 [decorators.py:305] Directly load AOT compilation from path /root/.cache/vllm/torch_compile_cache/torch_aot_compile/364deebc07fb29298a030d0ccc2e51dc97cb384b3e76521da7ac217f400d3e58/rank_0_0/model
INFO 05-02 20:47:28 [monitor.py:53] torch.compile took 7.62 s in total
INFO 05-02 20:47:29 [monitor.py:81] Initial profiling/warmup run took 0.25 s
INFO 05-02 20:47:30 [gpu_model_runner.py:5963] Profiling CUDA graph memory: PIECEWISE=19 (largest=128), FULL=11 (largest=64)
INFO 05-02 20:47:32 [gpu_model_runner.py:6042

## 8. Generate responses (same loop as starter)

Checkpoint file: `results/dev_results.responses.jsonl` — delete it to regenerate all dev answers.

In [9]:
CHUNK_SIZE = min(100, max(1, len(data)))

try:
    _results_dir = DRIVE_PROJECT_ROOT / "results"
except NameError:
    _results_dir = Path(OUTPUT_PATH).parent

_results_dir.mkdir(parents=True, exist_ok=True)
response_checkpoint = _results_dir / (Path(OUTPUT_PATH).stem + ".responses.jsonl")
print(f"Checkpoint path: {response_checkpoint}")

completed: dict = {}
if response_checkpoint.exists():
    with open(response_checkpoint) as f:
        for line in f:
            r = json.loads(line)
            completed[r["id"]] = r["response"]
    print(f"Resumed from checkpoint: {len(completed)} responses already generated")

remaining = [d for d in data if d.get("id") not in completed]
print(f"Generating {len(remaining)} remaining / {len(data)} total...")

for chunk_start in range(0, len(remaining), CHUNK_SIZE):
    chunk = remaining[chunk_start : chunk_start + CHUNK_SIZE]

    prompts = []
    for item in chunk:
        system, user = build_prompt(item["question"], item.get("options"))
        prompt_text = tokenizer.apply_chat_template(
            [{"role": "system", "content": system},
             {"role": "user",   "content": user}],
            tokenize=False,
            add_generation_prompt=True,
        )
        prompts.append(prompt_text)

    with _jupyter_stdout_for_vllm():
        outputs = llm.generate(prompts, sampling_params=sampling_params_free)

    with open(response_checkpoint, "a") as f:
        for item, out in zip(chunk, outputs):
            response = out.outputs[0].text.strip()
            completed[item["id"]] = response
            f.write(json.dumps({"id": item["id"], "response": response}) + "\n")

    done = len(completed)
    print(f"  {done}/{len(data)}  ({done / len(data) * 100:.1f}%)")

responses = [completed[d["id"]] for d in data]
print(f"Done. {len(responses)} responses ready.")

Rendering prompts:   0%|          | 0/100 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/100 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Checkpoint path: /content/drive/MyDrive/CSE151B/results/dev_results.responses.jsonl
Generating 112 remaining / 112 total...


Rendering prompts:   0%|          | 0/12 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/12 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  100/112  (89.3%)
  112/112  (100.0%)
Done. 112 responses ready.


## 9. Score responses (same as starter)

In [13]:
_judger_override = os.environ.get("CSE151B_JUDGER_DIR", "").strip()
JUDGER_ROOT = _judger_override or str(REPO_ROOT)

sys.path.insert(0, os.path.abspath(JUDGER_ROOT))
from judger import Judger
judger = Judger(strict_extract=False)

In [14]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring: 100%|██████████| 112/112 [00:07<00:00, 15.13it/s]

Scoring complete. 112 results.


## 10. Summary

In [15]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("DEV SET RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

DEV SET RESULTS
  MCQ        :   20 /   37  (54.05%)
  Free-form  :   41 /   75  (54.67%)
  Overall    :   61 /  112  (54.46%)


## 11. Save results

Same schema as the starter (`id`, `is_mcq`, `gold`, `response`, `correct`). Uses Drive when `DRIVE_PROJECT_ROOT` exists.

In [ ]:
SAVE_EVAL = True

try:
    out_path = DRIVE_PROJECT_ROOT / "results" / Path(OUTPUT_PATH).name
except NameError:
    out_path = Path(OUTPUT_PATH)

out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path.resolve()}")

## Next step

When satisfied on the dev slice, run **`starter_code_cse151b_comp.ipynb`** on full `data/public.jsonl` for comparable metrics. For **private** predictions as CSV, run **`notebooks/submission.ipynb`**.